In [1]:
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import pyarrow as pa
import polars as pl
from matplotlib import pyplot as plt
from deltalake import DeltaTable, write_deltalake

In [2]:
sys.path.append(str("/root/capsule/src/"))

In [3]:
from connects_common_connectivity.models import Cluster, ClusterMembership, CellToClusterMapping, MappingSet
from connects_common_connectivity.arrow_utils import build_arrow_schema, models_to_table, attach_linkml_metadata

In [4]:
# Realized that some of these should probably be CellToClusterMapping rather than ClusterMembership entries,
# i.e., the Patch-seq T-type assignments & the WNM MET-type assignments


# Load data sets & data items

In [5]:
# uses output of `patchseq_and_wnm_datasets` notebook

In [6]:
PATH = "../results/dataset/"
dataset_df = pl.read_delta(PATH)

In [7]:
dataset_df.select("id")

id
str
"""visp_exc_wnm"""
"""visp_exc_patchseq"""
"""visp_inh_patchseq"""
"""tasic_2018_visp_scrnaseq"""


In [8]:
PATH = "../results/dataitem/"
dataitem_df = pl.read_delta(PATH)

In [9]:
PATH = "../results/dataitem_dataset_association/"
di_ds_assoc_df = pl.read_delta(PATH)

# Load taxonomy information

In [10]:
# uses output of `process_patchseq_taxonomy_info` notebook

PATH = "../results/cluster/"
cluster_df = pl.read_delta(PATH)

In [11]:
cluster_df.select("project_id").unique()

project_id
str
"""tasic_2018_visp_scrnaseq"""
"""visp_met_types"""


# T-type mapping assignments

### Inh cells

In [12]:
ref_project_id = "tasic_2018_visp_scrnaseq"
project_id = "visp_inh_patchseq"

In [13]:
inh_ps_ttype_mapping_set = MappingSet(
    id="visp_inh_patchseq_ttype_mapping",
    name="visp_inh_patchseq_ttype_mapping",
    description="Mapping of t-types to VISp inhibitory Patch-seq neurons",
    method_name="Tree mapping",
    source_dataset="tasic_2018_visp_scrnaseq",
    target_dataset="visp_inh_patchseq",
    project_id=project_id,
)

In [14]:
schema = build_arrow_schema(MappingSet)
table = models_to_table([inh_ps_ttype_mapping_set], schema)
table = attach_linkml_metadata(table, linkml_class="MappingSet")  # version auto-populated

In [15]:
PATH = "../results/mappingset/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id"])

In [16]:
inh_ttype_df = pd.read_csv("../data/visp-features-and-mapping/patchseq_tx_cell_ttype_labels.csv", index_col=0)

In [17]:
dataitem_df.filter(pl.col("id").is_in([str(inh_ttype_df.index[0])]).alias("in_list")).shape[0] > 0

True

In [18]:
def create_self_and_parent_cell_cluster_mappings(
        my_id, my_type, project_id, ref_project_id,
        mapping_set, cluster_df, probability=None):
    """ Create CellToClusterMapping for a cluster and all parents in the taxonomic hierarchy

    Note: has (overly) simplistic assignment of probability
    """
    
    mapping_list = []
    
    has_parent = True
    while has_parent:
        mapping_list.append(CellToClusterMapping(
            id=f"{my_id}-{my_type}-{project_id}-{ref_project_id}", # not sure what ID is supposed to be for this, but seems to be required
            source_cell=my_id,
            target_cluster=my_type,
            project_id=project_id,
            probability=probability,
            mapping_set=mapping_set,
        ))

        my_parent = cluster_df.filter(
            (pl.col("id") == my_type) & (pl.col("project_id") == ref_project_id)
        ).select("parent").item()
        if my_parent is not None:
            has_parent = True
            my_type = my_parent
            probability = None # currently only recording probability for specific cluster call, not propagating to parent
        else:
            has_parent = False
    return mapping_list

In [19]:
cl_map_memb_list = []

In [20]:
for i, r in inh_ttype_df.iterrows():
    # Confirm that cell is a known DataItem
    if dataitem_df.filter(
        pl.col("id").is_in([str(inh_ttype_df.index[0])])
        ).shape[0] == 0:
        continue
    my_ttype = r.ttype

    cl_map_memb_list += create_self_and_parent_cell_cluster_mappings(
        str(i),
        my_ttype,
        project_id,
        ref_project_id,
        mapping_set=inh_ps_ttype_mapping_set.id,
        cluster_df=cluster_df,
    )


In [21]:
schema = build_arrow_schema(CellToClusterMapping)
table = models_to_table(cl_map_memb_list, schema)
table = attach_linkml_metadata(table, linkml_class="CellToClusterMapping")  # version auto-populated

In [22]:
PATH = "../results/celltoclustermapping/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id"])

### Exc cells

In [23]:
project_id = "visp_exc_patchseq"

In [24]:
exc_ps_ttype_mapping_set = MappingSet(
    id="visp_exc_patchseq_ttype_mapping",
    name="visp_exc_patchseq_ttype_mapping",
    description="Mapping of t-types to VISp excitatory Patch-seq neurons",
    method_name="Tree mapping",
    source_dataset="tasic_2018_visp_scrnaseq",
    target_dataset="visp_exc_patchseq",
    project_id=project_id,
)

In [25]:
exc_type_df = pd.read_csv("../data/visp-features-and-mapping/inferred_met_types.csv", index_col=0)

In [26]:
cl_map_memb_list = []

In [27]:
for i, r in exc_type_df.iterrows():
    my_ttype = r.t_type
    
    # translate to prior naming scheme (old = PT, new = ET)
    my_ttype = my_ttype.replace("ET", "PT")
    
    cl_map_memb_list += create_self_and_parent_cell_cluster_mappings(
        str(i),
        my_ttype,
        project_id,
        ref_project_id,
        mapping_set=exc_ps_ttype_mapping_set.id,
        cluster_df=cluster_df,
    )


In [28]:
schema = build_arrow_schema(CellToClusterMapping)
table = models_to_table(cl_map_memb_list, schema)
table = attach_linkml_metadata(table, linkml_class="CellToClusterMapping")  # version auto-populated

In [29]:
PATH = "../results/celltoclustermapping/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id"])

# MET-types

In [30]:
project_id = "visp_inh_patchseq"
ref_project_id = "visp_met_types"

In [31]:
def create_self_and_parent_cluster_memberships(
        my_id, my_type, project_id, ref_project_id,
        cluster_df):
    """ Create ClusterMembership for a cluster and all parents in the taxonomic hierarchy """
    
    membership_list = []
    
    has_parent = True
    while has_parent:
        membership_list.append(ClusterMembership(
            item=my_id,
            cluster=my_type,
            project_id=project_id,
        ))

        my_parent = cluster_df.filter(
            (pl.col("id") == my_type) & (pl.col("project_id") == ref_project_id)
        ).select("parent").item()
        if my_parent is not None:
            has_parent = True
            my_type = my_parent
        else:
            has_parent = False
    return membership_list

### Inh cells

In [32]:
inh_mettype_df = pd.read_csv("../data/visp-features-and-mapping/visp_met_cell_assignments_text_names.csv", index_col=0)

In [33]:
cl_memb_list = []

In [34]:
for i, r in inh_mettype_df.iterrows():
    my_mettype = r.met_type
    
    cl_memb_list += create_self_and_parent_cluster_memberships(
        str(i), my_mettype, project_id, ref_project_id, cluster_df)


In [35]:
schema = build_arrow_schema(ClusterMembership)
table = models_to_table(cl_memb_list, schema)
table = attach_linkml_metadata(table, linkml_class="ClusterMembership")  # version auto-populated

In [36]:
PATH = "../results/clustermembership/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id"])

### Exc cells

In [37]:
project_id = "visp_exc_patchseq"

In [38]:
cl_memb_list = []

In [39]:
for i, r in exc_type_df.dropna(subset=["met_type"]).iterrows():
    my_mettype = r.met_type
    
    cl_memb_list += create_self_and_parent_cluster_memberships(
        str(i), my_mettype, project_id, ref_project_id, cluster_df)

In [40]:
schema = build_arrow_schema(ClusterMembership)
table = models_to_table(cl_memb_list, schema)
table = attach_linkml_metadata(table, linkml_class="ClusterMembership")  # version auto-populated

In [41]:
PATH = "../results/clustermembership/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id"])

### WNM exc cells

In [42]:
project_id = "visp_exc_wnm"

In [43]:
exc_wnm_mettype_mapping_set = MappingSet(
    id="visp_exc_wnm_mettype_mapping",
    name="visp_exc_wnm_mettype_mapping",
    description="Mapping of met-types to VISp excitatory whole-neuron morphology neurons",
    method_name="Routed random forest mapping",
    source_dataset="visp_exc_patchseq",
    target_dataset="visp_exc_wnm",
    project_id=project_id,
)

In [44]:
wnm_exc_mettype_df = pd.read_csv("../data/visp-features-and-mapping/FullMorphMetaData_Master.csv", index_col=0)
wnm_exc_mettype_df["id"] = [s[:-4] for s in wnm_exc_mettype_df.index]

In [45]:
cl_map_memb_list = []

In [46]:
for i, r in wnm_exc_mettype_df.dropna(subset=["predicted_met_type"]).iterrows():
    my_mettype = r.predicted_met_type

    cl_map_memb_list += create_self_and_parent_cell_cluster_mappings(
        str(r.id),
        my_mettype,
        project_id,
        ref_project_id,
        mapping_set=exc_wnm_mettype_mapping_set.id,
        cluster_df=cluster_df,
        probability=r.probability,
    )

In [47]:
schema = build_arrow_schema(CellToClusterMapping)
table = models_to_table(cl_map_memb_list, schema)
table = attach_linkml_metadata(table, linkml_class="CellToClusterMapping")  # version auto-populated

In [48]:
PATH = "../results/celltoclustermapping/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id"])

# Read back

In [49]:
# Cluster memberships

PATH = "../results/clustermembership/"
cm_df = pl.read_delta(PATH)

cm_df

item,cluster,membership_score,probability,distance,project_id
str,str,f64,f64,f64,str
"""1039273993""","""L6b""",null,null,null,"""visp_exc_patchseq"""
"""1039273993""","""Glutamatergic""",null,null,null,"""visp_exc_patchseq"""
"""1039273993""","""cell""",null,null,null,"""visp_exc_patchseq"""
"""823218199""","""L5 ET-3""",null,null,null,"""visp_exc_patchseq"""
"""823218199""","""Glutamatergic""",null,null,null,"""visp_exc_patchseq"""
…,…,…,…,…,…
"""993245688""","""GABAergic""",null,null,null,"""visp_inh_patchseq"""
"""993245688""","""cell""",null,null,null,"""visp_inh_patchseq"""
"""993283588""","""Sncg-MET-1""",null,null,null,"""visp_inh_patchseq"""


In [50]:
# Cell to cluster mappings

PATH = "../results/celltoclustermapping/"
ccm_df = pl.read_delta(PATH)

ccm_df

id,mapping_set,source_cell,target_cluster,score,probability,notes,project_id
str,str,str,str,f64,f64,str,str
"""182709_6984-X2452-Y12423_reg-L…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""L5 ET-2""",null,0.988,null,"""visp_exc_wnm"""
"""182709_6984-X2452-Y12423_reg-G…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""Glutamatergic""",null,null,null,"""visp_exc_wnm"""
"""182709_6984-X2452-Y12423_reg-c…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""cell""",null,null,null,"""visp_exc_wnm"""
"""182709_7126-X2913-Y10535_reg-L…","""visp_exc_wnm_mettype_mapping""","""182709_7126-X2913-Y10535_reg""","""L5 ET-3""",null,0.918,null,"""visp_exc_wnm"""
"""182709_7126-X2913-Y10535_reg-G…","""visp_exc_wnm_mettype_mapping""","""182709_7126-X2913-Y10535_reg""","""Glutamatergic""",null,null,null,"""visp_exc_wnm"""
…,…,…,…,…,…,…,…
"""738925092-cell-visp_inh_patchs…","""visp_inh_patchseq_ttype_mappin…","""738925092""","""cell""",null,null,null,"""visp_inh_patchseq"""
"""772422057-Pvalb Th Sst-visp_in…","""visp_inh_patchseq_ttype_mappin…","""772422057""","""Pvalb Th Sst""",null,null,null,"""visp_inh_patchseq"""
"""772422057-Pvalb-visp_inh_patch…","""visp_inh_patchseq_ttype_mappin…","""772422057""","""Pvalb""",null,null,null,"""visp_inh_patchseq"""


In [51]:
ccm_df.filter(pl.col("mapping_set") == "visp_exc_wnm_mettype_mapping")

id,mapping_set,source_cell,target_cluster,score,probability,notes,project_id
str,str,str,str,f64,f64,str,str
"""182709_6984-X2452-Y12423_reg-L…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""L5 ET-2""",null,0.988,null,"""visp_exc_wnm"""
"""182709_6984-X2452-Y12423_reg-G…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""Glutamatergic""",null,null,null,"""visp_exc_wnm"""
"""182709_6984-X2452-Y12423_reg-c…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""cell""",null,null,null,"""visp_exc_wnm"""
"""182709_7126-X2913-Y10535_reg-L…","""visp_exc_wnm_mettype_mapping""","""182709_7126-X2913-Y10535_reg""","""L5 ET-3""",null,0.918,null,"""visp_exc_wnm"""
"""182709_7126-X2913-Y10535_reg-G…","""visp_exc_wnm_mettype_mapping""","""182709_7126-X2913-Y10535_reg""","""Glutamatergic""",null,null,null,"""visp_exc_wnm"""
…,…,…,…,…,…,…,…
"""220309_6744-X9496-Y18547_reg-G…","""visp_exc_wnm_mettype_mapping""","""220309_6744-X9496-Y18547_reg""","""Glutamatergic""",null,null,null,"""visp_exc_wnm"""
"""220309_6744-X9496-Y18547_reg-c…","""visp_exc_wnm_mettype_mapping""","""220309_6744-X9496-Y18547_reg""","""cell""",null,null,null,"""visp_exc_wnm"""
"""220315_6442-X7542-Y17434_reg-L…","""visp_exc_wnm_mettype_mapping""","""220315_6442-X7542-Y17434_reg""","""L4 IT""",null,1.0,null,"""visp_exc_wnm"""
